# Columnar contracts, policy expressions and liquidity-based partial fills

Second design notebook: **dummy data**, realistic typed boundaries, one account,
three listings, long-only positions, configurable resolution (default 1s).

Market state is conceptually `N × F × D`. We expose an actual fixed array
`feature_tensor: Array(Float64, shape=(3,2))` per listing, plus named Struct
columns used by rules. Structs preserve names and types; features need not have
equal dimensionality. A dense tensor requires a declared dimension ordering and
units (and padding/masks if dimensions differ).

**A strategy is rules/policies over market AND execution-state columns.** Market
conditions produce proposals; held quantity, cost, pending acquisition, and cash
determine which actions can be taken. Pure Polars expressions formulate both.
One Python loop advances time; no ticker loop, UDF, NumPy or Pandas calculation.

Scope: a volume-participation approximation, not exchange queue reconstruction
or parity with a numbered strategy. No live database access, fees, leverage,
latency randomness, or production durability is introduced.


## Contracts and transition ordering

| Table | Key | Contract |
|---|---|---|
| Source liquidity | build/attempt/listing/100ms bucket | Projection of ARTE-style fields; integer prices use 1e-4 scale; eligible execution volume/notional |
| Coverage | listing | Dummy certified build/attempt, complete start/end range; absent bucket is an error, explicit zero volume is valid |
| Market | listing/bar | Typed price, momentum, liquidity Structs; UTC start/end/available clocks; fixed feature array |
| State | listing | Held quantity, total book cost, stop/target; one pending BUY/SELL slot, remaining quantity and submission clock |
| Account | account_id | Cash and realized P&L; no leverage; a shared allocation policy |
| Evidence | listing/bar | Requests, volume/cash constrained fills, cancellation and next action |

Ticker is a display label; stable `listing_id` is the state key. This dummy session
has static identity mapping. Actual inputs need point-in-time identity and real
coverage certification. The source below is a **schema projection**, not the full
ARTE table; generated dummy momentum is a separate pretend producer feature.

At bar close: (1) book fills for actions submitted no later than this bar's start;
(2) evaluate rules on current completed features and post-fill positions;
(3) commit next pending actions. Whole-bar eligible volume is known only at close,
so fills are booked at close at volume-weighted execution price. A decision made
at this close cannot use this same bar's volume for a fill.

One participation budget per listing: `floor(participation * execution_volume)`.
With one pending slot, BUY and SELL never independently consume the same budget.
Across listings, sales fund buys, then cash-constrained buys scale proportionally
and round down. Remainders persist. Exit cancels unfilled acquisition and sells
only held shares. Stop/target are checked at completed close, with stop priority;
initial thresholds are pinned to the **first fill**, not recomputed on later fills.
Flat listings may re-enter whenever entry rules pass; no cooldown is modeled.
At the end, holdings and pending remainders stay visible; no terminal fill is invented.


In [ ]:
from datetime import datetime, timezone
import polars as pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(13)
RESOLUTION_MS = 1_000
PRICE_SCALE = 10_000
ORIGIN = datetime(2026, 10, 1, 8, 0, tzinfo=timezone.utc)
BUILD = "dummy-liquidity-build-v1"
ATTEMPT = "00000000-0000-0000-0000-000000000001"

# [1,K] policy snapshot: broadcast with cross joins; pinned identity travels in evidence.
policy = pl.DataFrame({"policy_id": ["close-momentum-v1"], "participation": [0.5],
    "entry_shares": [10], "stop_fraction": [0.05], "target_fraction": [0.10]})
initial_account = pl.DataFrame({"account_id": ["demo"], "cash": [600.0], "realized_pnl": [0.0]})
identity = pl.DataFrame({"listing_id": [101,102,103], "ticker": ["AAA","BBB","CCC"]},
                       schema={"listing_id": pl.UInt64,"ticker":pl.String})
initial_state = identity.with_columns(
    pl.when(pl.col("listing_id") == 101).then(4).otherwise(0).cast(pl.Int64).alias("quantity"),
    pl.when(pl.col("listing_id") == 101).then(400.0).otherwise(0.0).alias("book_cost"),
    pl.when(pl.col("listing_id") == 101).then(95.0).otherwise(0.0).alias("stop"),
    pl.when(pl.col("listing_id") == 101).then(110.0).otherwise(0.0).alias("target"),
    pl.lit("NONE").alias("pending_side"), pl.lit(0,dtype=pl.Int64).alias("remaining"),
    pl.lit(None,dtype=pl.Datetime("us","UTC")).alias("submitted_at"),
)

# [120*N,K] synthetic 100ms liquidity. Monetary price representation mirrors ARTE's 1e-4 scale.
# Every bucket exists in this dummy certified grid; CCC has explicitly ZERO liquidity in second 1.
raw = pl.DataFrame({"bucket_index":pl.int_range(0,120,eager=True,dtype=pl.UInt32)})
raw = raw.join(identity,how="cross").with_columns(
    pl.lit(BUILD).alias("build_id"),pl.lit(ATTEMPT).alias("attempt_id"),
    pl.lit(100,dtype=pl.UInt32).alias("resolution_ms"),
    (pl.col("bucket_index")//10).alias("second"),
).with_columns(
    pl.when(pl.col("listing_id")==101).then(pl.when(pl.col("second")<2).then(100).otherwise(94))
      .when(pl.col("listing_id")==102).then(pl.when(pl.col("second")<4).then(50).otherwise(46))
      .otherwise(pl.when(pl.col("second")<3).then(40).otherwise(46))
      .mul(PRICE_SCALE).cast(pl.UInt64).alias("close_int"),
    pl.when((pl.col("listing_id")==103)&(pl.col("second")==1)).then(0.0)
      .when((pl.col("listing_id")==103)&(pl.col("second")==3)).then(0.8)
      .otherwise(0.4).alias("execution_volume"),
).with_columns(
    (pl.col("execution_volume")*pl.col("close_int")/PRICE_SCALE).alias("execution_notional"),
    pl.lit(1,dtype=pl.UInt8).alias("price_valid"),
).drop("second")
coverage = identity.select("listing_id").with_columns(
    pl.lit(BUILD).alias("build_id"),pl.lit(ATTEMPT).alias("attempt_id"),
    pl.lit(0,dtype=pl.UInt32).alias("first_bucket"),
    pl.lit(119,dtype=pl.UInt32).alias("last_bucket"),
    pl.lit(True).alias("coverage_complete"),
)
print(raw.head(6))
print(initial_state)


## Build market columns and the tensor view

The array dimension contract is F=3 features, D=2 dimensions:
`price=[completed_close_dollars, execution_vwap_dollars]`,
`momentum=[line, signal]`, `liquidity=[eligible_shares, eligible_notional_dollars]`.
These are heterogeneous units, deliberately unnormalized. The tensor is an
inspectable representation, not an ML-ready encoding. Null VWAP with zero volume
is masked by the execution capacity; it must not become a synthetic fill.


In [ ]:
def make_market(raw, coverage, resolution_ms):
    assert type(resolution_ms) is int and resolution_ms>=100 and resolution_ms%100==0
    assert raw.select(pl.struct("listing_id","bucket_index").n_unique()).item()==raw.height
    assert coverage.select(pl.col("listing_id").n_unique()).item()==coverage.height
    assert coverage.select(pl.col("coverage_complete").all()).item()
    assert coverage.join(raw.select("listing_id").unique(),on="listing_id",how="anti").is_empty()
    checked = raw.join(coverage,on=["listing_id","build_id","attempt_id"],how="left",validate="m:1")
    assert checked.select(pl.col("coverage_complete").is_not_null().all()).item()
    assert checked.select(((pl.col("bucket_index")>=pl.col("first_bucket"))
                           &(pl.col("bucket_index")<=pl.col("last_bucket"))
                           &(pl.col("resolution_ms")==100)).all()).item()
    counts=checked.group_by("listing_id").agg(pl.len().alias("rows"),
        pl.col("first_bucket").first(),pl.col("last_bucket").first())
    assert counts.select((pl.col("rows")==pl.col("last_bucket")-pl.col("first_bucket")+1).all()).item()
    assert raw.select(((pl.col("execution_volume")>=0)&pl.col("execution_volume").is_finite()
                       &(pl.col("execution_notional")>=0)&pl.col("execution_notional").is_finite()
                       &(((pl.col("execution_volume")==0)&(pl.col("execution_notional")==0))
                         |((pl.col("execution_volume")>0)&(pl.col("execution_notional")>0)))
                       &(pl.col("price_valid").is_in([0,1]))
                       &((pl.col("price_valid")==0)|(pl.col("close_int")>0))).all()).item()
    width=resolution_ms//100
    flat=(raw.sort(["listing_id","bucket_index"])
        .with_columns((pl.col("bucket_index")//width).alias("bar"))
        .group_by(["listing_id","ticker","build_id","attempt_id","bar"]).agg(
            pl.col("close_int").last(),pl.col("price_valid").min(),
            pl.col("execution_volume").sum(),pl.col("execution_notional").sum(),
            pl.len().alias("source_rows"))
        .with_columns(
            (pl.lit(ORIGIN)+pl.duration(milliseconds=pl.col("bar")*resolution_ms)).alias("bar_start"),
            (pl.lit(ORIGIN)+pl.duration(milliseconds=(pl.col("bar")+1)*resolution_ms)).alias("bar_end"),
            pl.when(pl.col("execution_volume")>0)
              .then(pl.col("execution_notional")/pl.col("execution_volume")).otherwise(None).alias("vwap"),
            # Dummy producer-owned momentum, sampled only at the completed boundary.
            pl.when((pl.col("bar")*resolution_ms>=4000)&(pl.col("listing_id")==102))
              .then(-0.1).otherwise(0.2).alias("line"),pl.lit(0.05).alias("signal"),
        ).sort(["bar","listing_id"]))
    assert flat.select((pl.col("source_rows")==width).all()).item(),"Partial resolution bar is not certified"
    market=flat.with_columns(
        pl.col("bar_end").alias("available_at"),
        pl.struct(pl.col("close_int"),pl.col("price_valid").cast(pl.Boolean).alias("valid")).alias("price"),
        pl.struct("line","signal").alias("momentum"),
        pl.struct(pl.col("execution_volume").alias("volume"),
                  pl.col("execution_notional").alias("notional"),pl.col("vwap")).alias("liquidity"),
        # Nested fixed arrays: each row has shape [F=3,D=2]; N rows form [N,3,2].
        pl.concat_list(
            pl.concat_list(pl.col("close_int")/PRICE_SCALE,pl.col("vwap")).list.to_array(2),
            pl.concat_list("line","signal").list.to_array(2),
            pl.concat_list("execution_volume","execution_notional").list.to_array(2),
        ).list.to_array(3).alias("feature_tensor"),
    ).select("listing_id","ticker","build_id","attempt_id","bar","bar_start","bar_end",
             "available_at","price","momentum","liquidity","feature_tensor")
    assert market.schema["feature_tensor"]==pl.Array(pl.Float64,shape=(3,2))
    return market

market=make_market(raw,coverage,RESOLUTION_MS)
print(market.head(3))
print(market.schema)


## Strategy definition: named expression rules plus typed policy parameters

This is the executable definition, not prose that the engine secretly interprets.
Rule outputs remain visible. Price validity gates EVERY decision rule. Entry
requires a flat position, no pending action, positive MACD line above signal.
Exit precedence is stop -> target -> negative momentum. A pending SELL continues;
an exit can cancel a pending BUY. New entries request the policy's quantity.


In [ ]:
def field(feature, dimension):
    # Expression builder only: returns a native Polars Expr, never a row callback.
    return pl.col(feature).struct.field(dimension)

RULES={
    "entry_momentum":(field("momentum","line")>field("momentum","signal"))
                       &(field("momentum","line")>0),
    "stop_hit":field("price","close_int")/PRICE_SCALE<=pl.col("stop"),
    "target_hit":field("price","close_int")/PRICE_SCALE>=pl.col("target"),
    "momentum_failure":field("momentum","line")<field("momentum","signal"),
}

def propose_actions(rows):
    decisions=rows.with_columns(
        RULES["entry_momentum"].alias("entry_momentum"),
        RULES["stop_hit"].alias("stop_hit"),RULES["target_hit"].alias("target_hit"),
        RULES["momentum_failure"].alias("momentum_failure"),
    ).with_columns(
        pl.when((pl.col("pending_side")=="SELL")&(pl.col("post_remaining")>0)).then(pl.lit("KEEP_SELL"))
          .when((pl.col("post_quantity")>0)&field("price","valid")&pl.col("stop_hit")).then(pl.lit("EXIT_STOP"))
          .when((pl.col("post_quantity")>0)&field("price","valid")&pl.col("target_hit")).then(pl.lit("EXIT_TARGET"))
          .when((pl.col("post_quantity")>0)&field("price","valid")&pl.col("momentum_failure")).then(pl.lit("EXIT_MOMENTUM"))
          .when(pl.col("post_remaining")>0).then(pl.lit("KEEP_BUY"))
          .when((pl.col("post_quantity")==0)&field("price","valid")&pl.col("entry_momentum")).then(pl.lit("ENTER"))
          .otherwise(pl.lit("HOLD")).alias("action"),
    ).with_columns(pl.col("action").is_in(["EXIT_STOP","EXIT_TARGET","EXIT_MOMENTUM"]).alias("exit_requested"))
    return decisions


## Approximate execution: partial fills as a columnar transition

Capacity is floor(participation * eligible volume). SELL is also bounded by held
quantity. BUY requests are bounded by capacity first, then shared cash after sales.
VWAP describes the whole completed execution interval, not its opening quote.
This does not claim passive limit execution or queue priority. Available volume
is a modeled participation resource, not guaranteed exchange liquidity.


In [ ]:
def execute_pending(state,account,current_market):
    rows=state.join(current_market.drop("ticker"),on="listing_id",validate="1:1").join(policy,how="cross")
    rows=rows.with_columns(
        (field("liquidity","volume")*pl.col("participation")).floor().cast(pl.Int64).alias("capacity"),
        field("liquidity","vwap").fill_null(0.0).alias("fill_price"),
        ((pl.col("remaining")>0)&(pl.col("submitted_at")<=pl.col("bar_start"))
         &(field("liquidity","volume")>0)).fill_null(False).alias("eligible"),
        pl.when(pl.col("quantity")>0).then(pl.col("book_cost")/pl.col("quantity"))
          .otherwise(0.0).alias("average_cost"),
    ).with_columns(
        pl.when(pl.col("eligible")&(pl.col("pending_side")=="SELL"))
          .then(pl.min_horizontal("remaining","capacity","quantity")).otherwise(0).alias("sell_fill"),
        pl.when(pl.col("eligible")&(pl.col("pending_side")=="BUY"))
          .then(pl.min_horizontal("remaining","capacity")).otherwise(0).alias("buy_candidate"),
    ).with_columns(
        (pl.col("sell_fill")*pl.col("fill_price")).alias("sale_cash"),
        (pl.col("buy_candidate")*pl.col("fill_price")).alias("requested_cash"),
        (pl.col("sell_fill")*(pl.col("fill_price")-pl.col("average_cost"))).alias("realized_delta"),
    )
    # [N] requests -> [1] total budget -> [N] simultaneous allocation.
    budget=account.join(rows.select(pl.col("sale_cash").sum(),
        pl.col("requested_cash").sum().alias("total_requested_cash")),how="cross"
    ).with_columns((pl.col("cash")+pl.col("sale_cash")).alias("available_cash"))
    rows=rows.join(budget,how="cross").with_columns(
        pl.when(pl.col("total_requested_cash")>0)
          .then(pl.min_horizontal(pl.lit(1.0),pl.col("available_cash")/pl.col("total_requested_cash")))
          .otherwise(0.0).alias("cash_scale"),
    ).with_columns((pl.col("buy_candidate")*pl.col("cash_scale")).floor().cast(pl.Int64).alias("buy_fill"))
    rows=rows.with_columns(
        (pl.col("quantity")+pl.col("buy_fill")-pl.col("sell_fill")).alias("post_quantity"),
        (pl.col("remaining")-pl.col("buy_fill")-pl.col("sell_fill")).alias("post_remaining"),
        (pl.col("book_cost")+pl.col("buy_fill")*pl.col("fill_price")
         -pl.col("sell_fill")*pl.col("average_cost")).alias("post_book_cost"),
        (pl.col("sale_cash")-pl.col("buy_fill")*pl.col("fill_price")).alias("cash_delta"),
    ).with_columns(
        pl.when((pl.col("quantity")==0)&(pl.col("buy_fill")>0))
          .then(pl.col("fill_price")*(1-pl.col("stop_fraction")))
          .otherwise(pl.col("stop")).alias("stop"),
        pl.when((pl.col("quantity")==0)&(pl.col("buy_fill")>0))
          .then(pl.col("fill_price")*(1+pl.col("target_fraction")))
          .otherwise(pl.col("target")).alias("target"),
    )
    next_account=account.join(rows.select(pl.col("cash_delta").sum(),pl.col("realized_delta").sum()),how="cross"
    ).select("account_id",(pl.col("cash")+pl.col("cash_delta")).alias("cash"),
             (pl.col("realized_pnl")+pl.col("realized_delta")).alias("realized_pnl"))
    fills=rows.select("listing_id","bar","bar_start","bar_end","submitted_at","policy_id",
        "pending_side","remaining","capacity","fill_price","eligible","cash_scale",
        "buy_candidate","buy_fill","sell_fill","post_remaining","cash_delta")
    return rows,next_account,fills


In [ ]:
def commit(decisions,account):
    decisions=decisions.with_columns(
        pl.when(pl.col("exit_requested")).then(pl.lit("SELL"))
          .when(pl.col("action")=="ENTER").then(pl.lit("BUY"))
          .when(pl.col("post_remaining")>0).then(pl.col("pending_side"))
          .otherwise(pl.lit("NONE")).alias("next_side"),
        pl.when(pl.col("exit_requested")).then(pl.col("post_quantity"))
          .when(pl.col("action")=="ENTER").then(pl.col("entry_shares"))
          .otherwise(pl.col("post_remaining")).cast(pl.Int64).alias("next_remaining"),
        pl.when(pl.col("exit_requested")&(pl.col("pending_side")=="BUY"))
          .then(pl.col("post_remaining")).otherwise(0).alias("cancelled_buy_quantity"),
    )
    state=decisions.select("listing_id","ticker",pl.col("post_quantity").alias("quantity"),
        pl.when(pl.col("post_quantity")>0).then(pl.col("post_book_cost")).otherwise(0.0).alias("book_cost"),
        pl.when(pl.col("post_quantity")>0).then(pl.col("stop")).otherwise(0.0).alias("stop"),
        pl.when(pl.col("post_quantity")>0).then(pl.col("target")).otherwise(0.0).alias("target"),
        pl.col("next_side").alias("pending_side"),pl.col("next_remaining").alias("remaining"),
        pl.when(pl.col("exit_requested")|(pl.col("action")=="ENTER")).then(pl.col("bar_end"))
          .when(pl.col("next_remaining")>0).then(pl.col("submitted_at")).otherwise(None).alias("submitted_at"),
    ).sort("listing_id")
    account_snapshot=account.join(decisions.select(
        (pl.col("post_quantity")*field("price","close_int")/PRICE_SCALE).sum().alias("market_value"),
        pl.col("post_book_cost").sum().alias("book_cost")),how="cross"
    ).with_columns((pl.col("cash")+pl.col("market_value")).alias("equity"),
                   (pl.col("market_value")-pl.col("book_cost")).alias("unrealized_pnl"))
    actions=decisions.select("listing_id","bar","available_at","policy_id","post_quantity",
        "entry_momentum","stop_hit","target_hit","momentum_failure","action",
        "cancelled_buy_quantity","next_side","next_remaining")
    return state,account_snapshot,actions

def run(market,starting_account=initial_account):
    state,account=initial_state,starting_account
    fill_batches,action_batches,state_batches,account_batches=[],[],[],[]
    for current in market.sort(["bar","listing_id"]).partition_by("bar",maintain_order=True):
        assert current.height==state.height
        assert current.select(pl.col("listing_id").n_unique()).item()==state.height
        assert state.join(current,on="listing_id",how="anti").is_empty()
        assert current.select((pl.col("available_at")==pl.col("bar_end")).all()).item()
        rows,account,fills=execute_pending(state,account,current)
        state,snapshot,actions=commit(propose_actions(rows),account)
        assert account.select((pl.col("cash")>=-1e-8).all()).item()
        assert state.select(((pl.col("quantity")>=0)&(pl.col("remaining")>=0)).all()).item()
        clock=current.select(pl.col("bar").first())
        fill_batches.append(fills);action_batches.append(actions)
        state_batches.append(state.join(clock,how="cross"));account_batches.append(snapshot.join(clock,how="cross"))
    return {"fills":pl.concat(fill_batches),"actions":pl.concat(action_batches),
            "states":pl.concat(state_batches),"accounts":pl.concat(account_batches)}

result=run(market)
print("Rules and actions")
print(result["actions"].filter(~pl.col("action").is_in(["HOLD","KEEP_BUY","KEEP_SELL"])))
print("Partial fill evidence")
print(result["fills"].filter(pl.col("remaining")>0))
print("Account snapshots")
print(result["accounts"])


## Checks that make the contracts reviewable

Checks cover independent accounting identities, carried remainders, no execution
on zero liquidity, volume budgets, cancellation, causal clocks, and missing source
coverage rejection. Future-tail changes must not change prior facts. Missing
coverage is distinct from an explicit zero-volume row. Dummy VWAP is derived from
dummy eligible volume/notional; it does not assert real-market representativeness.


In [ ]:
cash_check=result["accounts"].join(result["fills"].group_by("bar").agg(pl.col("cash_delta").sum())
    .sort("bar").with_columns(pl.col("cash_delta").cum_sum().alias("cash_change")),on="bar")
assert cash_check.select(((pl.col("cash")-600-pl.col("cash_change")).abs()<1e-8).all()).item()
assert result["accounts"].select(((pl.col("equity")-1000-pl.col("realized_pnl")
                                 -pl.col("unrealized_pnl")).abs()<1e-8).all()).item()
assert result["fills"].select(((pl.col("buy_fill")+pl.col("sell_fill")<=pl.col("capacity"))
                              &(pl.col("post_remaining")>=0)).all()).item()
actual=result["fills"].filter(pl.col("buy_fill")+pl.col("sell_fill")>0)
assert actual.select((pl.col("submitted_at")<=pl.col("bar_start")).all()).item()
assert result["fills"].filter(pl.col("capacity")==0).select(
    ((pl.col("buy_fill")+pl.col("sell_fill"))==0).all()).item()
assert result["fills"].filter((pl.col("buy_fill")>0)&(pl.col("post_remaining")>0)).height>0
assert result["actions"].filter(pl.col("cancelled_buy_quantity")>0).height>0

# Position conservation, independently derived from fill deltas per stable identity.
q=result["fills"].sort(["listing_id","bar"]).with_columns(
    (pl.col("buy_fill")-pl.col("sell_fill")).cum_sum().over("listing_id").alias("delta"))
q=q.join(initial_state.select("listing_id",pl.col("quantity").alias("initial")),on="listing_id")
q=q.join(result["states"].select("listing_id","bar","quantity"),on=["listing_id","bar"])
assert q.select((pl.col("quantity")==pl.col("initial")+pl.col("delta")).all()).item()

changed=market.with_columns(pl.when(pl.col("bar")>=6)
    .then(pl.struct((field("price","close_int")+100_000).alias("close_int"),field("price","valid").alias("valid")))
    .otherwise(pl.col("price")).alias("price"))
# Keep the tensor projection consistent with the changed named market fields.
changed=changed.with_columns(pl.concat_list(
    pl.concat_list(field("price","close_int")/PRICE_SCALE,field("liquidity","vwap")).list.to_array(2),
    pl.concat_list(field("momentum","line"),field("momentum","signal")).list.to_array(2),
    pl.concat_list(field("liquidity","volume"),field("liquidity","notional")).list.to_array(2),
).list.to_array(3).alias("feature_tensor"))
changed_result=run(changed)
assert result["states"].filter(pl.col("bar")<6).equals(changed_result["states"].filter(pl.col("bar")<6))
assert result["fills"].filter(pl.col("bar")<6).equals(changed_result["fills"].filter(pl.col("bar")<6))
assert result["accounts"].filter(pl.col("bar")<6).equals(changed_result["accounts"].filter(pl.col("bar")<6))
scarce=run(market,initial_account.with_columns(pl.lit(40.0).alias("cash")))
assert scarce["fills"].filter((pl.col("buy_candidate")>0)&(pl.col("cash_scale")<1)).height>0
assert scarce["accounts"].select((pl.col("cash")>=-1e-8).all()).item()
# Descending key sort permutes whole rows, including nested tensor arrays.
assert result["states"].equals(run(market.sort(["bar","listing_id"],descending=True))["states"])
try:
    make_market(raw.filter(~((pl.col("listing_id")==103)&(pl.col("bucket_index")==10))),coverage,RESOLUTION_MS)
except AssertionError:
    print("PASS: missing source coverage rejected; explicit zero volume retained")
else:
    raise AssertionError("Missing certified bucket was accepted")
print("PASS: tensor contract, accounting, quantity conservation, partial remainders,")
print("participation capacity, zero-liquidity behavior, exit cancellation and causal clocks/tail.")


## Design boundary

The named market Structs are authoritative for this prototype's policy evaluator;
`feature_tensor` is a derived view, not a separate decision authority. Execution
uses the liquidity Struct, never the tensor's price dimensions blindly.
Real integration would select certified ARTE products/builds, align clocks and
identity, and apply the same table contracts. Multiple strategies/accounts/orders
would require explicit grouped capital and liquidity allocation. An expression
graph can formulate that policy, but changing allocation changes the model.

The next discussion should be about feature schemas, policy composition, allocation,
and close-based versus intrabar modeling—not copying detailed broker mechanics.


## Pressure test: N listings from start time to end time

Edit the configuration below and run these cells in order. The interval is UTC,
start-inclusive/end-exclusive, and must contain whole resolution bars.
`N_LISTINGS × number_of_bars × 3 × 2` is the logical feature-matrix size.

The generator creates completed-resolution liquidity/feature rows directly,
using deterministic Polars expressions. It avoids materializing a 100ms source
for this **compute pressure test**. It is not a test of ClickHouse reading,
canonical source preparation, or producer certification. All execution and
strategy transitions reuse `execute_pending`, `propose_actions`, and `commit`.

**Full backtest wall time** includes input validation, sorting, sequential clock
advancement, partial fills, cash allocation, policy rules, state updates,
invariant checks, summary collection and final result assembly. Data generation
and display are separately timed/excluded. This is one cold measured run; no
hidden warm-up or averaged performance claim is made.

By default, retain final state plus account/counter history (O(N+T) outputs).
Set `KEEP_FULL_HISTORY=True` to also retain all fill/action/state rows (O(NT)),
as the earlier `run` does. Both execute every boundary and ticker; only output
retention changes. The report identifies the mode. Memory guards reject the
requested size rather than truncate it. The estimate is a conservative planning
heuristic, not a measured peak-RSS guarantee.


In [ ]:
from time import perf_counter
from datetime import timedelta
import platform

N_LISTINGS = 1_000
START_TIME = "2026-10-01T08:00:00+00:00"
END_TIME = "2026-10-01T08:02:00+00:00"
PRESSURE_RESOLUTION_MS = 1_000
STARTING_CASH = 100_000.0
KEEP_FULL_HISTORY = False
MAX_MARKET_ROWS = 2_000_000
MAX_ESTIMATED_GIB = 1.0

pressure_start = datetime.fromisoformat(START_TIME)
pressure_end = datetime.fromisoformat(END_TIME)
assert pressure_start.tzinfo is not None and pressure_end.tzinfo is not None, "Timezone required"
pressure_start = pressure_start.astimezone(timezone.utc)
pressure_end = pressure_end.astimezone(timezone.utc)
assert type(N_LISTINGS) is int and N_LISTINGS>0
assert type(PRESSURE_RESOLUTION_MS) is int and PRESSURE_RESOLUTION_MS>=100 and PRESSURE_RESOLUTION_MS%100==0
assert 0<STARTING_CASH<float("inf")
duration = pressure_end-pressure_start
duration_us = (duration.days*86400+duration.seconds)*1_000_000+duration.microseconds
assert duration_us>0 and duration_us%(PRESSURE_RESOLUTION_MS*1000)==0, "Use a positive whole-bar interval"
N_BARS = duration_us//(PRESSURE_RESOLUTION_MS*1000)
assert N_BARS<=2**32-1, "bar index contract is UInt32"
MARKET_ROWS = N_LISTINGS*N_BARS
# Typed feature/identity/clock storage plus intermediate joins and output headroom.
ESTIMATED_GIB = MARKET_ROWS*384*(8 if KEEP_FULL_HISTORY else 3)/(1024**3)
if MARKET_ROWS>MAX_MARKET_ROWS or ESTIMATED_GIB>MAX_ESTIMATED_GIB:
    raise MemoryError(f"Requested {MARKET_ROWS:,} rows; estimated working set {ESTIMATED_GIB:.2f} GiB. "
                      "Reduce N/time range, or deliberately raise the explicit guards.")
print(f"Feature shape: [{N_LISTINGS:,}, {N_BARS:,}, 3, 2] (N, time, features, dimensions)")
print(f"UTC interval: [{pressure_start.isoformat()}, {pressure_end.isoformat()})")
print(f"Estimated working set: {ESTIMATED_GIB:.3f} GiB; full history: {KEEP_FULL_HISTORY}")


In [ ]:
def make_pressure_inputs(n_listings,n_bars,start,resolution_ms,starting_cash):
    listings=pl.DataFrame({"listing_id":pl.int_range(1,n_listings+1,eager=True,dtype=pl.UInt64)})
    listings=listings.with_columns(pl.concat_str(pl.lit("DUMMY-"),
        pl.col("listing_id").cast(pl.String).str.pad_start(6,"0")).alias("ticker"))
    clock=pl.DataFrame({"bar":pl.int_range(0,n_bars,eager=True,dtype=pl.UInt32)})
    flat=clock.join(listings,how="cross").with_columns(
        pl.lit("dummy-pressure-build-v1").alias("build_id"),pl.lit(ATTEMPT).alias("attempt_id"),
        (pl.lit(start)+pl.duration(milliseconds=pl.col("bar").cast(pl.Int64)*resolution_ms)).alias("bar_start"),
        (pl.lit(start)+pl.duration(milliseconds=(pl.col("bar").cast(pl.Int64)+1)*resolution_ms)).alias("bar_end"),
        # Repeating, phase-shifted prices/momentum exercise entries, exits and remainders.
        ((20.0+(pl.col("listing_id")%200).cast(pl.Float64)*0.1
          +((pl.col("bar").cast(pl.Int64)+(pl.col("listing_id")%7).cast(pl.Int64))%16-8)*0.03)
         *PRICE_SCALE).round(0).cast(pl.UInt64).alias("close_int"),
        pl.when((pl.col("listing_id")+pl.col("bar"))%5==0).then(0.0)
          .otherwise((20+(pl.col("listing_id")%20)).cast(pl.Float64)*resolution_ms/1000)
          .alias("execution_volume"),
        pl.when((pl.col("listing_id")+pl.col("bar"))%12<6).then(0.2).otherwise(-0.1).alias("line"),
        pl.lit(0.05).alias("signal"),
    ).with_columns(
        (pl.col("execution_volume")*pl.col("close_int")/PRICE_SCALE).alias("execution_notional"),
        pl.when(pl.col("execution_volume")>0).then(pl.col("close_int")/PRICE_SCALE)
          .otherwise(None).alias("vwap"),
    )
    generated=flat.with_columns(
        pl.col("bar_end").alias("available_at"),
        pl.struct("close_int",pl.lit(True).alias("valid")).alias("price"),
        pl.struct("line","signal").alias("momentum"),
        pl.struct(pl.col("execution_volume").alias("volume"),
                  pl.col("execution_notional").alias("notional"),"vwap").alias("liquidity"),
        pl.concat_list(
            pl.concat_list(pl.col("close_int")/PRICE_SCALE,"vwap").list.to_array(2),
            pl.concat_list("line","signal").list.to_array(2),
            pl.concat_list("execution_volume","execution_notional").list.to_array(2),
        ).list.to_array(3).alias("feature_tensor"),
    ).select(market.columns)
    state=listings.with_columns(
        pl.lit(0,dtype=pl.Int64).alias("quantity"),pl.lit(0.0).alias("book_cost"),
        pl.lit(0.0).alias("stop"),pl.lit(0.0).alias("target"),pl.lit("NONE").alias("pending_side"),
        pl.lit(0,dtype=pl.Int64).alias("remaining"),
        pl.lit(None,dtype=pl.Datetime("us","UTC")).alias("submitted_at"),
    )
    account=pl.DataFrame({"account_id":["pressure-demo"],"cash":[starting_cash],"realized_pnl":[0.0]})
    return generated,state,account

generation_started=perf_counter()
pressure_market,pressure_state,pressure_account=make_pressure_inputs(
    N_LISTINGS,N_BARS,pressure_start,PRESSURE_RESOLUTION_MS,STARTING_CASH)
GENERATION_SECONDS=perf_counter()-generation_started
print(f"Generated {pressure_market.height:,} market rows in {GENERATION_SECONDS:.3f}s")
print(f"Market buffers: {pressure_market.estimated_size('mb'):.1f} MiB (not process peak memory)")


In [ ]:
def timed_backtest(market_input,starting_state,starting_account,keep_full_history=False):
    # Start BEFORE validation/sort; stop AFTER assembling all retained outputs.
    started=perf_counter()
    state,account=starting_state,starting_account
    n=state.height
    assert n>0 and state.select(pl.col("listing_id").n_unique()).item()==n
    assert market_input.height>0 and market_input.height%n==0
    assert market_input.schema==market.schema, "Use the same typed market contract"
    ordered=market_input.sort(["bar","listing_id"])
    assert ordered.select(pl.struct("listing_id","bar").n_unique()).item()==ordered.height
    n_bars=ordered.height//n
    account_batches,stat_batches=[],[]
    fill_batches,action_batches,state_batches=[],[],[]
    # Slice contiguous N-row matrices; do not allocate a list of every bar first.
    for offset in range(0,ordered.height,n):
        current=ordered.slice(offset,n)
        assert current.select(pl.col("bar").n_unique()).item()==1
        assert state.join(current,on="listing_id",how="anti").is_empty()
        assert current.select((pl.col("available_at")==pl.col("bar_end")).all()).item()
        rows,account,fills=execute_pending(state,account,current)
        state,snapshot,actions=commit(propose_actions(rows),account)
        assert account.select((pl.col("cash")>=-1e-8).all()).item()
        assert state.select(((pl.col("quantity")>=0)&(pl.col("remaining")>=0)).all()).item()
        clock=current.select(pl.col("bar").first())
        account_batches.append(snapshot.join(clock,how="cross"))
        stat_batches.append(fills.select(
            (pl.col("buy_fill")+pl.col("sell_fill")).sum().alias("filled_shares"),
            ((pl.col("buy_fill")+pl.col("sell_fill"))>0).sum().alias("fill_rows"),
            (pl.col("post_remaining")>0).sum().alias("remaining_rows")))
        if keep_full_history:
            fill_batches.append(fills);action_batches.append(actions)
            state_batches.append(state.join(clock,how="cross"))
    histories={"accounts":pl.concat(account_batches),"statistics":pl.concat(stat_batches),
               "final_state":state,"final_account":account}
    if keep_full_history:
        histories.update(fills=pl.concat(fill_batches),actions=pl.concat(action_batches),states=pl.concat(state_batches))
    elapsed=perf_counter()-started
    totals=histories["statistics"].select(pl.col("filled_shares").sum(),pl.col("fill_rows").sum())
    report=pl.DataFrame({"listings":[n],"bars":[n_bars],"ticker_bar_rows":[ordered.height],
        "full_backtest_seconds":[elapsed],"ticker_bars_per_second":[ordered.height/elapsed],
        "seconds_per_boundary":[elapsed/n_bars],"keep_full_history":[keep_full_history],
        "input_buffer_mib":[market_input.estimated_size('mb')]})
    return histories,report.join(totals,how="cross")

# Small equivalence check is separate from the measured pressure run.
reference_history,_=timed_backtest(market,initial_state,initial_account,True)
assert reference_history["states"].equals(result["states"])
assert reference_history["fills"].equals(result["fills"])
assert reference_history["actions"].equals(result["actions"])
assert reference_history["accounts"].equals(result["accounts"])
print("PASS: pressure runner matches the earlier full-history backtest")

pressure_result,pressure_report=timed_backtest(
    pressure_market,pressure_state,pressure_account,KEEP_FULL_HISTORY)
pressure_report=pressure_report.with_columns(
    pl.lit(GENERATION_SECONDS).alias("generation_seconds"),
    pl.lit(pl.__version__).alias("polars_version"),pl.lit(pl.thread_pool_size()).alias("polars_threads"),
    pl.lit(platform.python_version()).alias("python_version"),
    pl.lit(pressure_start).alias("start_time"),pl.lit(pressure_end).alias("end_time"),
    pl.lit(PRESSURE_RESOLUTION_MS).alias("resolution_ms"),
)
print("Full backtest benchmark (generation and printing excluded)")
print(pressure_report.transpose(include_header=True))
print("Last account snapshot")
print(pressure_result["accounts"].tail(1))
assert pressure_report.select(pl.col("ticker_bar_rows")==MARKET_ROWS).item()
assert pressure_result["accounts"].height==N_BARS
assert pressure_report.select(pl.col("filled_shares")>0).item(), "Benchmark must exercise fills"


Scale N or the interval gradually and inspect `full_backtest_seconds` and
`ticker_bars_per_second`. Change `PRESSURE_RESOLUTION_MS` to measure a different
number of causal boundaries. Report output-retention mode when comparing runs.
For full-history mode the timed interval includes concatenation of all evidence.
This measures the eager Polars formulation as written, including Python clock
orchestration; it does not promise a particular production speedup.

The default does not write files. If saving benchmark reports or histories, use
`D:/TradingML/runtimes/vectorized_backtest/`, never the repository. Free the large
objects or restart the kernel before another large trial to avoid retaining both
trials in memory. Market buffer size is reported; process peak RSS is not measured.
